# Sections
Section plots of temperature and salinity compared to observations.

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from pyprojroot import here
from ruamel.yaml import YAML

from blomster.grid import read_strait_indices


In [ ]:
# get project root
root_path = Path(here())
yaml=YAML()
with open(root_path.joinpath('notebooks/diagnostics.yml'), "r") as f:
    cfg = yaml.load(f)

# info of diagnostic tool and generated webpages
work_root = cfg['blomster']['work_root']
www_root = cfg['jupyterbook']['www_root']

# info of experiments
caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
grid_label = cfg['experiment']['grid_label']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']

# info of grids
grid_file = cfg['grids'][grid_label]['grid_file']
region_file = cfg["grids"][grid_label]["region_file"]
section_file = cfg["grids"][grid_label]["section_file"]
grid_file = cfg['grids'][grid_label]['grid_file']

# info of observational datasets
obs_file = cfg['datasets']['PHC3.0']['ANNUAL_file']

# info of model climatologies
climo_file1 = cfg['climos']['blom']['hy']['ANN']
climo_file2 = cfg['climos']['blom']['hm']['ANN']

print(f"Observational dataset: {obs_file}")
#print(f"Model climatology 1: {climo_file1}")
#print(f"Model climatology 2: {climo_file2}")

## Section map
Location of sections

In [ ]:
# set the section names to be plotted
section_names = ['fram_strait']

with xr.open_dataset(grid_file) as ds:
    plat = ds['plat']
    plon = ds['plon']

# make plot showing the location of the section points
fig = plt.figure(figsize=(6, 6))
ax = plt.axes(projection=ccrs.NorthPolarStereo())
ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.coastlines()
ax.gridlines(linestyle=':', color='gray')

for section_name in section_names:
    iidx, jidx, _, _ = read_strait_indices(section_file, section_name)
    seclat = plat.isel(
        y=xr.DataArray(jidx, dims="points"),
        x=xr.DataArray(iidx, dims="points"),
    )
    seclon = plon.isel(
        y=xr.DataArray(jidx, dims="points"),
        x=xr.DataArray(iidx, dims="points"),
    )
    ax.plot(
        seclon, seclat, marker='o', markersize=3, linestyle='-',
        color='r', transform=ccrs.PlateCarree(), label=section_name,
    )

ax.set_title('Location of sections')
ax.legend(loc='lower left', fontsize=8)
plt.show()


## Section plots

In [ ]:
with xr.open_dataset(grid_file) as ds:
    pdepth = ds['pdepth']
    plat = ds['plat']
    plon = ds['plon']

# mapping of variable names from PHC to BLOM
varmappings = {'temp': 'templvl', 'salt': 'salnlvl'}
cmapmapping = {'temp': 'thermal', 'salt': 'haline'}
datasource = {'temp': 'thermal', 'salt': 'haline'}

# option 1: selective sections
section_names = ['fram_strait']

for section_name in section_names:
    iidx, jidx, _, _ = read_strait_indices(section_file, section_name)
#   k = 0   # order of plots in the subplots
    for varobs, varblom in varmappings.items():
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 4.5))
        # read BLOM data and compute section
        try:
            ds = xr.open_dataset(climo_file1)
        except (FileNotFoundError):
            try:
                ds = xr.open_dataset(climo_file2)
            except FileNotFoundError:
                print(f"{climo_file2} not found either, stop...")
                raise

        da = ds[varblom].isel(time=0)
        section = da.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
        seclat = plat.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
        seclon = plon.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
        secdepth = pdepth.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )

        # set points with lat/lon values, and find the maxium depth
        botdepth = secdepth.max().values

        # read PHC data and select the section point by point
        with xr.open_dataset(obs_file) as ds:
            da_obs = ds[varobs]
            da_obs = da_obs.assign_coords(lon=(((da_obs.lon + 180) % 360) - 180)).sortby("lon")

            # PHC longitudes on the same convention as the model section
            lon_pts = ((seclon + 180) % 360) - 180
            section_obs = da_obs.sel(
                lat=xr.DataArray(seclat.values, dims="points"),
                lon=xr.DataArray(lon_pts.values, dims="points"),
                method="nearest",
            ).assign_coords(points=section["points"])

        # get min, max values within certain quantile
        vals = np.concatenate([section.values.ravel(), section_obs.values.ravel()])
        vmin, vmax = np.nanquantile(vals, [0.001, 0.999])

        vmin1, vmax1 = np.nanquantile(section.values.ravel(), [0.001, 0.999])
        vmin2, vmax2 = np.nanquantile(section_obs.values.ravel(), [0.001, 0.999])

        # print simple statistics of the section data
        df = pd.DataFrame(
            {
                "min": [vmin1, vmin2],
                "max": [vmax1, vmax2],
            },
            index=["NorESM", "Obs"],
        )

        print(f"variable: {varblom}")
        print("--- --- --- --- --- --- --- ---")
        print(df.to_string(justify="center"))
        print("--- --- --- --- --- --- --- ---")

        # plot the sections
        section_obs.plot(ax=axs[0], x="points", y="depth", ylim=[0, botdepth], vmin=vmin, vmax=vmax, cmap=f"cmo.{cmapmapping[varobs]}")
        section.plot(ax=axs[1], x="points", y="depth", ylim = [0, botdepth], vmin=vmin, vmax=vmax, cmap=f"cmo.{cmapmapping[varobs]}")

        axs[0].set_title(f'PHC3.0: {section_obs.standard_name} across {section_name}')
        axs[1].set_title(f'NorESM: {section.standard_name} across {section_name}')
        axs[0].invert_yaxis()
        axs[1].invert_yaxis()

        plt.show()
